In [ ]:
import os

import numpy as np
import cv2
import torch
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from scipy.ndimage import binary_closing, binary_opening, binary_erosion

import transformers

from PIL import Image

# Window Detection

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

processor = transformers.SegformerImageProcessor.from_pretrained("nvidia/segformer-b5-finetuned-ade-640-640")
model = transformers.AutoModelForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-640-640")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)

In [ ]:
# Generate masked images for windows
window_class_ids = [8, 21]
base_data_path = './data/temp'

image_files = []
for folder_name in os.listdir(base_data_path):
    folder_path = os.path.join(base_data_path, folder_name)
    if os.path.isdir(folder_path):
        image_path = os.path.join(folder_path, 'image.jpg')
        if os.path.exists(image_path):
            image_files.append((folder_name, image_path))

print(f"Found {len(image_files)} images to process.")

for folder_name, image_path in image_files:
    image = Image.open(image_path).convert("RGB")

    # Process the image and run the model
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get the segmentation map and resize it to the original image size
    logits = outputs.logits # Shape: (1, num_classes, H, W)
    segmentation_map = logits.argmax(dim=1).squeeze().cpu().numpy() # Shape: (H, W)
    
    # Create the binary mask for the window class
    # The mask will be True where the pixel class is 21, and False otherwise
    window_mask = np.isin(segmentation_map, window_class_ids)
    
    # Convert the boolean mask to a saveable image format (0 for background, 255 for window)
    mask_image = Image.fromarray((window_mask * 255).astype(np.uint8))
    
    # Resize mask to original image dimensions before saving
    mask_image = mask_image.resize(image.size, Image.NEAREST)

    # Save the mask
    mask_filename = 'mask.png'
    mask_image.save(os.path.join(base_data_path, folder_name, mask_filename))

print("Finished generating all window masks.")

In [ ]:
# apply erosion to the masks

ORIGINAL_DEPTH_FILENAME = 'depth.png' 

# 3. Set the filename for the masks created by your script.
MASK_FILENAME = 'mask.png'

# 4. Set the desired filename for the NEW corrected depth maps.
CORRECTED_DEPTH_FILENAME = 'depth_corrected.png'

# 5. Fine-tune the erosion setting if needed.
EROSION_ITERATIONS = 3
# --- End of Configuration ---


# --- Main Processing Logic ---
# Get a list of all subdirectories in the base data path
folder_names = [f for f in os.listdir(base_data_path) if os.path.isdir(os.path.join(base_data_path, f))]

print(f"Found {len(folder_names)} scene folders to process.")

# Loop through each scene folder using tqdm for a progress bar
for folder_name in tqdm(folder_names, desc="Correcting depth maps"):
    folder_path = os.path.join(base_data_path, folder_name)
    
    # Define the full paths for the necessary files
    depth_path = os.path.join(folder_path, ORIGINAL_DEPTH_FILENAME)
    mask_path = os.path.join(folder_path, MASK_FILENAME)
    corrected_depth_path = os.path.join(folder_path, CORRECTED_DEPTH_FILENAME)

    # Skip this folder if the original depth map or the mask doesn't exist
    if not os.path.exists(depth_path) or not os.path.exists(mask_path):
        continue

    # Load the mask and the original depth map
    # The mask is converted to a boolean array (True/False)
    window_mask = np.array(Image.open(mask_path).convert("L")) > 0
    # cv2.IMREAD_UNCHANGED ensures depth data is read with its original bit depth
    depth_map = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)

    # If the mask is all black (no window detected), just copy the original depth map
    if not np.any(window_mask):
        cv2.imwrite(corrected_depth_path, depth_map)
        continue

    # --- Core Correction Logic ---
    # Erode the mask to find the boundary pixels (the wall just around the window)
    eroded_mask = binary_erosion(window_mask, iterations=EROSION_ITERATIONS)
    boundary_mask = window_mask & ~eroded_mask
    
    # Get the depth values from the boundary region
    # Also, filter out any invalid depth pixels (e.g., zero values)
    valid_wall_pixels = depth_map[boundary_mask & (depth_map > 0)]
    
    if len(valid_wall_pixels) == 0:
        # As a fallback, if no valid wall pixels are found, use the original depth map
        corrected_depth = depth_map
    else:
        # Calculate the median depth of the wall (more robust to outliers than average)
        wall_depth_value = np.median(valid_wall_pixels)
        
        # Create a copy of the depth map to modify
        corrected_depth = depth_map.copy()
        
        # Fill the entire window region with the calculated wall depth
        corrected_depth[window_mask] = wall_depth_value

    # Save the new, corrected depth map into the same folder
    cv2.imwrite(corrected_depth_path, corrected_depth)

print(f"\n✅ Finished correcting depth maps. Check for '{CORRECTED_DEPTH_FILENAME}' in each subfolder.")

# Direct Depth Correction

In [ ]:
base_data_path = './data/temp'

ORIGINAL_DEPTH_FILENAME = 'depth.png' 
CORRECTED_DEPTH_FILENAME = 'depth_corrected.png'

THRESHOLD_PERCENTILE = 95 # 95 means the top 5% of depth values will be considered part of a window.
CLEANING_ITERATIONS = 5
EROSION_ITERATIONS = 10

folder_names = [f for f in os.listdir(base_data_path) if os.path.isdir(os.path.join(base_data_path, f))]

print(f"Found {len(folder_names)} scene folders to process using the threshold method.")

for folder_name in tqdm(folder_names, desc="Correcting depth maps"):
    folder_path = os.path.join(base_data_path, folder_name)
    
    depth_path = os.path.join(folder_path, ORIGINAL_DEPTH_FILENAME)
    corrected_depth_path = os.path.join(folder_path, CORRECTED_DEPTH_FILENAME)

    if not os.path.exists(depth_path):
        continue

    depth_map = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    
    if depth_map is None or depth_map.size == 0:
        continue
    
    # Get all valid depth values (non-zero) to calculate the threshold
    valid_depths = depth_map[depth_map > 0]
    
    # If there are no valid depth pixels, just copy the original and move on
    if valid_depths.size == 0:
        cv2.imwrite(corrected_depth_path, depth_map)
        continue

    depth_threshold = np.percentile(valid_depths, THRESHOLD_PERCENTILE)
    
    # Create a mask of all pixels exceeding threshold
    max_value = np.max(valid_depths)
    extreme_depth_mask = (depth_map > depth_threshold) | (depth_map == max_value)

    # Clean the mask to get a solid shape
    temp_mask = binary_closing(extreme_depth_mask, iterations=CLEANING_ITERATIONS)
    window_mask = binary_opening(temp_mask, iterations=CLEANING_ITERATIONS).astype(bool)

    if not np.any(window_mask):
        cv2.imwrite(corrected_depth_path, depth_map)
        continue

    eroded_mask = binary_erosion(window_mask, iterations=EROSION_ITERATIONS)
    boundary_mask = window_mask & ~eroded_mask
    
    valid_wall_pixels = depth_map[boundary_mask & (depth_map > 0)]
    
    if len(valid_wall_pixels) == 0:
        corrected_depth = depth_map
    else:
        wall_depth_value = np.median(valid_wall_pixels)
        corrected_depth = depth_map.copy()
        corrected_depth[window_mask] = wall_depth_value

    cv2.imwrite(corrected_depth_path, corrected_depth)

print(f"\n✅ Finished. Check for '{CORRECTED_DEPTH_FILENAME}' in each subfolder.")